# Real Time Speech to Translated Speech

Translating my own voice into a different language, as I speak it!

In [1]:
# # We ask Python: "What version are you?"
# import sys

# # Print the Python version so we can see it
# print(sys.version)


In [2]:
# # Upgrade pip
# !pip install -q --upgrade pip

# # Install only open-source Hugging Face + LangChain tools
!pip install -q \
langchain langchain-community langchain-huggingface \
transformers torch torchaudio \
soundfile av accelerate sentencepiece

# # Audio support
# !apt-get install -y ffmpeg

In [30]:
from google.colab import files
files.upload()

Saving 10504.mp3 to 10504.mp3


{'10504.mp3': b'ID3\x04\x00\x00\x00\x00\x00#TSSE\x00\x00\x00\x0f\x00\x00\x03Lavf58.29.100\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\xff\xfb\x94\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00Info\x00\x00\x00\x0f\x00\x00\x00\xd1\x00\x01;\x00\x00\x04\x07\t\x0c\x0e\x11\x13\x15\x18\x1a\x1d #%(*-/1469<?ADFIKNPRUX[]`begjlnquwy|~\x81\x83\x86\x88\x8a\x8d\x91\x93\x95\x98\x9a\x9d\x9f\xa2\xa4\xa7\xa9\xad\xaf\xb1\xb4\xb6\xb9\xbb\xbe\xc0\xc3\xc5\xc9\xcb\xce\xd0\xd2\xd5\xd7\xda\xdc\xdf\xe1\xe5\xe7\xea\xec\xee\xf1\xf3\xf6\xf8\xfb\xfd\x00\x00\x00\x00Lavc58.54\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x01;\x00\xdc\xb6R\x16\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x0

In [4]:
# !pip install huggingface_hub
# !pip install transformers
# !pip install langchain
# !pip install accelerate
# !pip install langchain langchain-community langchain-huggingface

In [5]:
from google.colab import userdata
sec_key=userdata.get('HF_TOKEN')


In [6]:
import os
os.environ["HUGGINGFACEHUB_API_TOKEN"] = sec_key
print(f"HUGGINGFACEHUB_API_TOKEN successfully set: {os.environ['HUGGINGFACEHUB_API_TOKEN'] != ''}")

HUGGINGFACEHUB_API_TOKEN successfully set: True


In [7]:
# Use a pipeline as a high-level helper
from transformers import pipeline

whisper = pipeline("automatic-speech-recognition", model="openai/whisper-large-v3")

Device set to use cuda:0


In [8]:
# from langchain_huggingface import HuggingFacePipeline
# # Assuming 'pipe' is a pre-defined transformers pipeline object
# hf = HuggingFacePipeline(pipeline=whisper)
# The HuggingFacePipeline wrapper is not suitable for ASR tasks in this context.
# We will use the 'whisper' pipeline directly for transcription.

In [9]:
#hf

In [10]:
transcription_result = whisper("10176.mp3")
print(transcription_result)

`return_token_timestamps` is deprecated for WhisperFeatureExtractor and will be removed in Transformers v5. Use `return_attention_mask` instead, as the number of frames can be inferred from it.
Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.
Transcription using a multilingual Whisper will default to language detection followed by transcription instead of translation to English. This might be a breaking change for your use case. If you want to instead always translate your audio to English, make sure to pass `language='en'`. See https://github.com/huggingface/transformers/pull/28687 for more details.


{'text': ' एक बार फिर सक्रिया क्राहक बन जाएं. Netflix कंपनी पुनह सक्रिया करण का'}


In [11]:
text = transcription_result['text']
print(f"Transcribed Text: {text}")

Transcribed Text:  एक बार फिर सक्रिया क्राहक बन जाएं. Netflix कंपनी पुनह सक्रिया करण का


In [12]:
# Use a pipeline as a high-level helper
from transformers import pipeline

translation = pipeline("translation", model="facebook/nllb-200-distilled-1.3B" , device=0)

Device set to use cuda:0


In [13]:
lang1to2 = translation(text, tgt_lang="eng_Latn", src_lang="hin_Deva", max_length=400)[0]['translation_text']

In [14]:
print(lang1to2)

become an active viewer again.


In [15]:
# Use a pipeline as a high-level helper
from transformers import pipeline

TTSmodel = pipeline("text-to-speech", model="suno/bark" , device=0)

Device set to use cuda:0


In [17]:
output=TTSmodel(lang1to2)

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:10000 for open-end generation.


# Option 1: Quick fix in the call
Update your generation line to include the pad_token_id. Bark usually uses the eos_token_id for padding, but you need to tell the pipeline to use an attention mask.

In [18]:
# Pass the eos_token_id as the pad_token_id to satisfy the requirement
output = TTSmodel(lang1to2, forward_params={"pad_token_id": 10000})


#OR

#Option 2: Pre-configure the Pipeline
If you want to avoid adding params every time you call it, you can try setting it during initialization (though Bark is a bit finicky with the standard pipeline setup):

In [19]:
# TTSmodel = pipeline("text-to-speech", model="suno/bark", device=0)
# TTSmodel.tokenizer.pad_token_id = 10000 # Manually set the pad token

In [20]:
import scipy.io.wavfile as wavfile

# Generate the audio
output = TTSmodel(lang1to2, forward_params={"pad_token_id": 10000})

# Extract the sampling rate and the audio data (numpy array)
# Bark typically outputs at 24000Hz
sampling_rate = output["sampling_rate"]
audio_data = output["audio"].squeeze() # Flatten the array from (1, N) to (N,)

# Save to a file
wavfile.write("output_audio.wav", rate=sampling_rate, data=audio_data)

print("Audio saved successfully as 'output_audio.wav'")

Audio saved successfully as 'output_audio.wav'


In [21]:
from IPython.display import Audio
Audio("output_audio.wav", autoplay=True)

#CREATING A FUNCTION TO DO VOICE TO VOICE TRANSLATION

# ---------- Language Maps ----------

In [22]:

LANGUAGE_MAP = {
    "English": "eng_Latn",
    "Hindi": "hin_Deva",
    "French": "fra_Latn",
    "German": "deu_Latn",
    "Spanish": "spa_Latn",
    "Chinese": "zho_Hans",
    "Japanese": "jpn_Jpan",
    "Korean": "kor_Hang",
    "Arabic": "arb_Arab",
    "Russian": "rus_Cyrl"
}

WHISPER_TO_NLLB = {
    "en": "eng_Latn",
    "hi": "hin_Deva",
    "fr": "fra_Latn",
    "de": "deu_Latn",
    "es": "spa_Latn",
    "zh": "zho_Hans",
    "ja": "jpn_Jpan",
    "ko": "kor_Hang",
    "ar": "arb_Arab",
    "ru": "rus_Cyrl"
}



# ---------- Detect Language + Transcribe ----------

In [23]:
import re

def is_hindi(text):
    # Check for Devanagari script (Hindi) characters
    devanagari_pattern = re.compile(r"[\u0900-\u097F]+")
    return bool(devanagari_pattern.search(text))

def detect_language_and_text(audio_path):
    """
    Uses Whisper pipeline from your notebook
    """
    result = whisper(audio_path)
    text = result["text"]
    # Whisper pipeline output doesn't seem to include 'language' key by default for this setup
    language = result.get("language", "unknown")

    # Fallback to text-based detection if Whisper doesn't provide language
    if language == "unknown" and text.strip():
        if is_hindi(text):
            language = "hi"
        # Add other language detections if needed

    return text, language

# ---------- Ask User for Target Language ----------

In [24]:

def choose_target_language():
    print("\nAvailable Target Languages:\n")

    languages = list(LANGUAGE_MAP.items())
    for i, (name, _) in enumerate(languages, start=1):
        print(f"{i}. {name}")

    while True:
        try:
            choice = int(input("\nEnter target language number: "))
            if 1 <= choice <= len(languages):
                return languages[choice - 1]
        except:
            pass
        print("❌ Invalid choice. Try again.")



## ---------- FINAL ONE-LINE FUNCTION ----------

In [25]:
import scipy.io.wavfile as wavfile
from IPython.display import Audio

def voice_to_voice_translate_auto(
    audio_path,
    output_path="translated_voice.wav"
):
    """
    Voice → Text → Auto Detect Language
    Ask Target Language
    Text → Translation → Voice
    """

    # 1️⃣ Detect language & transcribe
    text, whisper_lang = detect_language_and_text(audio_path)
    print(f"\nDetected Source Language: {whisper_lang}")
    print("Transcribed Text:", text)

    src_lang = WHISPER_TO_NLLB.get(whisper_lang)
    if src_lang is None:
        raise ValueError("❌ Detected language not supported")

    # 2️⃣ Choose target language
    tgt_name, tgt_lang = choose_target_language()
    print(f"\nTranslating to: {tgt_name}")

    # 3️⃣ Translate
    translated_text = translation(
        text,
        src_lang=src_lang,
        tgt_lang=tgt_lang,
        max_length=400
    )[0]["translation_text"]

    print("Translated Text:", translated_text)

    # 4️⃣ Text → Speech (using suno/bark model)
    speech = TTSmodel(
        translated_text,
        forward_params={"pad_token_id": 10000}
    )

    wavfile.write(
        output_path,
        speech["sampling_rate"],
        speech["audio"].squeeze() # Ensure audio is flattened
    )

    print(f"\n✅ Output audio saved as: {output_path}")

    # 5️⃣ 🔊 Auto-play audio in notebook
    display(Audio(output_path, autoplay=True))

    return {
        "detected_language": whisper_lang,
        "source_text": text,
        "translated_text": translated_text,
        "output_audio": output_path,
        "AUDIO": Audio(output_path, autoplay=True)
    }

In [31]:
voice_to_voice_translate_auto("10504.mp3")


Detected Source Language: hi
Transcribed Text:  पर पुरुषों, महिलाओं और बच्चों की जीवन बचाने के परती बड़े उच्छा से समर्पित है।

Available Target Languages:

1. English
2. Hindi
3. French
4. German
5. Spanish
6. Chinese
7. Japanese
8. Korean
9. Arabic
10. Russian

Enter target language number: 7

Translating to: Japanese
Translated Text: しかし,男性,女性,子供たちの命を救うことに専念しています.

✅ Output audio saved as: translated_voice.wav


{'detected_language': 'hi',
 'source_text': ' पर पुरुषों, महिलाओं और बच्चों की जीवन बचाने के परती बड़े उच्छा से समर्पित है।',
 'translated_text': 'しかし,男性,女性,子供たちの命を救うことに専念しています.',
 'output_audio': 'translated_voice.wav',
 'AUDIO': <IPython.lib.display.Audio object>}